# Sanjay MK2 Cloud ML Validation

Run this notebook in a cloud Jupyter runtime when you want heavier validation than normal CI should run. It is intended for Google Colab, JupyterLab on a GPU VM, or another Linux cloud notebook with Python 3.11/3.12.

Normal GitHub Actions CI stays lightweight. This notebook owns full pytest, torch-backed suites, YOLO checkpoint checks, optional scenario validation, and optional ONNX smoke tests.

## 1. Runtime Setup

If the notebook is already inside the repo, it uses the current checkout. If not, it clones `SANJAY_REPO_URL` and checks out `SANJAY_BRANCH`.

In [ ]:
from __future__ import annotations

import os
import platform
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("SANJAY_REPO_URL", "https://github.com/aniket-27507/Sanjay_MK2.git")
BRANCH = os.environ.get("SANJAY_BRANCH", "main")
REPO_DIR = Path(os.environ.get("SANJAY_REPO_DIR", "Sanjay_MK2")).resolve()

def run(cmd: list[str], check: bool = True, cwd: Path | None = None) -> subprocess.CompletedProcess:
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if not Path("src").exists():
    if not REPO_DIR.exists():
        run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)

print("cwd:", Path.cwd())
print("python:", sys.version)
print("platform:", platform.platform())

## 2. Install Dependencies

This is intentionally heavier than CI. Cloud notebooks should install the full project requirements so torch, Ultralytics, ONNX, and model validation tools are present.

In [ ]:
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])

## 3. Environment Check

This cell records the ML runtime before validation. CUDA is optional for tests, but useful for model validation and export checks.

In [ ]:
import numpy as np
import torch

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

## 4. Full Test Suite

Run the full suite here because this cloud runtime should not hit the local macOS Python 3.14 + torch/OpenMP crash. If it fails, keep the traceback as the cloud validation artifact.

In [ ]:
run([sys.executable, "-m", "pytest", "-q"])

## 5. Focused ML and Model Validation

Set these paths after uploading or mounting model artifacts. Missing weights are skipped so the notebook can still validate the environment and test suite.

In [ ]:
RGB_WEIGHTS = Path(os.environ.get("SANJAY_RGB_WEIGHTS", "runs/detect/police_full_v2/weights/best.pt"))
THERMAL_WEIGHTS = Path(os.environ.get("SANJAY_THERMAL_WEIGHTS", "runs/detect/thermal_police_v1/weights/best.pt"))
ONNX_MODEL = Path(os.environ.get("SANJAY_ONNX_MODEL", "runs/export/model.onnx"))
REPORT_DIR = Path(os.environ.get("SANJAY_REPORT_DIR", "reports/cloud_validation"))
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("RGB weights:", RGB_WEIGHTS, "exists=", RGB_WEIGHTS.exists())
print("Thermal weights:", THERMAL_WEIGHTS, "exists=", THERMAL_WEIGHTS.exists())
print("ONNX model:", ONNX_MODEL, "exists=", ONNX_MODEL.exists())
print("Report dir:", REPORT_DIR)

In [ ]:
if RGB_WEIGHTS.exists():
    run([
        sys.executable,
        "scripts/validate_model.py",
        "--yolo",
        str(RGB_WEIGHTS),
        "--all",
        "--compare",
        "--report",
        str(REPORT_DIR / "rgb_validation.json"),
    ])
else:
    print("Skipping RGB scenario validation: weights not found.")

In [ ]:
if THERMAL_WEIGHTS.exists():
    run([
        sys.executable,
        "scripts/validate_model.py",
        "--thermal",
        str(THERMAL_WEIGHTS),
        "--all",
        "--report",
        str(REPORT_DIR / "thermal_validation.json"),
    ])
else:
    print("Skipping thermal scenario validation: weights not found.")

## 6. Optional ONNX Smoke Test

This verifies that an exported ONNX artifact can be loaded by ONNX Runtime. It does not replace Jetson TensorRT benchmarking.

In [ ]:
if ONNX_MODEL.exists():
    import onnxruntime as ort

    session = ort.InferenceSession(str(ONNX_MODEL), providers=["CPUExecutionProvider"])
    print("ONNX inputs:", [(i.name, i.shape, i.type) for i in session.get_inputs()])
    print("ONNX outputs:", [(o.name, o.shape, o.type) for o in session.get_outputs()])
else:
    print("Skipping ONNX smoke test: model not found.")

## 7. Validation Notes

Expected outputs:
- Full pytest result from the cloud runtime.
- Optional `reports/cloud_validation/rgb_validation.json` when RGB weights are available.
- Optional `reports/cloud_validation/thermal_validation.json` when thermal weights are available.
- ONNX input/output metadata when an exported model is available.

Known limitation: scenario validation for real-photo-trained YOLO checkpoints still has the documented BEV sim-to-real gap. Use training-set validation metrics as the authoritative perception metric until photorealistic sim or real sensor validation is available.